# Value Iteration implementation on Frozen Lake environment

This project implements **Value Iteration** to solve the **Frozen Lake** environment, a classic discrete reinforcement learning problem.

## Create frozen lake environment

Now let's create a frozen lake environment class, it should have the following features:
1. User should be able to pass the grid (2D list that contains the information about the starting point, frozen cell, holes and the goal)
2. create a helper function to create a grid based on provided parameters.
3. user should determine whethere the action is going to be deterministic or stochastic based on slipper mode.
4. state will be determine by row and column state.
5. there should be a method that takes current state and action abd based on that it should return a list of dictionary that contain the following fields `prod, next_state, reward, is_terminated`

In [1]:
import random
import numpy as np

In [2]:
def generate_grid_randomly(n_rows, n_cols, n_holes=4):
    """
    Generate a random 2D Frozen Lake grid.

    The grid consists of:
    - 'F' : Frozen (safe) cells
    - 'H' : Holes (terminal failure states)
    - 'S' : Start state
    - 'G' : Goal state

    Args:
        n_rows (int): Number of rows in the grid.
        n_cols (int): Number of columns in the grid.
        n_holes (int, optional): Number of hole cells to place.
            Defaults to 4.

    Returns:
        list[list[str]]: A 2D list representing the Frozen Lake grid.
    """
    
    grid = [["F" for _ in range(n_cols)] for _ in range(n_rows)]    
    states = [(r, c) for r in range(n_rows) for c in range(n_cols)]
    
    random.shuffle(states)

    for i in range(0, n_holes):
        hole = states[i%(n_rows*n_cols)]
        r, c = hole
        grid[r][c] = "H"

    # start state
    r, c = states[-1]
    grid[r][c] = "S"
    
    # goal state
    r, c = states[-2]
    grid[r][c] = "G"

    return grid

In [3]:
grid = generate_grid_randomly(10, 10, n_holes=12)
grid

[['S', 'F', 'F', 'F', 'F', 'H', 'F', 'F', 'H', 'F'],
 ['F', 'F', 'F', 'F', 'H', 'F', 'F', 'F', 'F', 'F'],
 ['F', 'F', 'F', 'F', 'F', 'F', 'F', 'F', 'F', 'F'],
 ['F', 'H', 'H', 'F', 'F', 'F', 'F', 'H', 'F', 'F'],
 ['F', 'F', 'F', 'F', 'F', 'F', 'F', 'F', 'F', 'F'],
 ['F', 'H', 'F', 'F', 'F', 'F', 'F', 'F', 'F', 'F'],
 ['F', 'F', 'F', 'F', 'F', 'H', 'F', 'F', 'F', 'F'],
 ['F', 'F', 'F', 'F', 'H', 'F', 'F', 'F', 'F', 'F'],
 ['F', 'F', 'F', 'F', 'H', 'F', 'F', 'F', 'F', 'H'],
 ['F', 'F', 'G', 'H', 'F', 'F', 'F', 'F', 'F', 'F']]

Now let's create FrozenLakeEnvironment but before that we need to define some constants

In [4]:
ACTION_NAME_TO_IDX = {
    "left": 0,
    "down": 1,
    "right": 2,
    "up": 3
}
ACTION_IDX_TO_NAME = {v:k for k,v in ACTION_NAME_TO_IDX.items()}
ACTION_TO_STEP_MAPPER = {
    0: (0, -1), # left means staying in the same row (row unchanged), moving one column left from the current cell i.e -1
    1: (1, 0), # down means moving to the next row (+1 row) but staying in the same column
    2: (0, 1), # right means staying in the same row (row unchanged), moving to the subsequent right column i.e +1
    3: (-1, 0) # up means staying in the same column but moving one row up i.e -1 
}

In [114]:
class FrozenLakeEnvironment:
    def __init__(self, grid, slippery=True):
        self.grid = grid
        self.n_rows = len(grid)
        self.n_cols = len(grid[0])
        self.slippery = slippery
        self.allowed_actions = list(ACTION_NAME_TO_IDX.values())
        
        self.action_names_to_idx = ACTION_NAME_TO_IDX
        self.action_to_step_mapper = ACTION_TO_STEP_MAPPER
        
        self.start_state = self.__find("S")
        self.goal_state = self.__find("G")
        self.current_state = self.start_state
        

    def __find(self, value):
        """
        Find the position of a given value in the grid.

        Iterates through the grid and returns the first (row, column)
        coordinate where the specified value is found.
    
        Args:
            value (str): The grid cell value to search for
                (e.g., 'S', 'G', 'H', 'F').
    
        Returns:
            tuple[int, int] | None: The (row, column) position of the value
            if found; otherwise, None.
        
        """
        for r in range(self.n_rows):
            for c in range(self.n_cols):
                if self.grid[r][c] == value:
                    return (r, c)

    def __is_terminal(self, state):
        """
        check if the given state is terminating state or not (G or H)
        """
        r, c = state
        return self.grid[r][c] in ["G", "H"]

    def __move(self, state, action):
        """
        Compute the next state given a current state and an action.
    
        The agent is moved according to the action mapping. If the action
        would cause the agent to step outside the grid boundaries, the
        position is clipped to remain within valid limits.
    
        Args:
            state (tuple[int, int]): Current (row, column) position
                of the agent.
            action (int): Action index mapped to a movement direction
                via `self.action_to_step_mapper`.
    
        Returns:
            tuple[int, int]: The next (row, column) state after applying
            the action.
        """
        r, c = state
        step_r, step_c = self.action_to_step_mapper[action]
        new_r = min(max(0, r+step_r), self.n_rows -1) #make sure step is a valid one
        new_c = min(max(0, c+step_c), self.n_cols -1)
        return new_r, new_c
        
    def get_transition_prob(self, state, action):
        """
        Compute the transition dynamics for a given state-action pair.
    
        Returns a list of possible transitions, each defined by the
        transition probability, next state, reward, and terminal flag.
        The environment supports both deterministic and stochastic
        (slippery) dynamics.
    
        If the current state is terminal, the agent remains in the same
        state with probability 1 and receives zero reward.
    
        In slippery mode, the intended action succeeds with probability
        0.8, while the agent slips to the left or right action with
        probability 0.1 each.
    
        Args:
            state (tuple[int, int]): Current (row, column) state.
            action (int): Action index chosen by the agent.
    
        Returns:
            list[dict]: A list of transition dictionaries with keys:
                - "prob" (float): Transition probability.
                - "new_state" (tuple[int, int]): Resulting state.
                - "reward" (float): Reward received after the transition.
                - "game_over" (bool): Whether the next state is terminal.
        """
        
        if self.__is_terminal(state):
            return [{"prob": 1.0,
                     "new_state": state,
                     "reward": 0,
                     "game_over": True}]

        # --- deterministic action --------
        actions = [action]
        probs = [1.0]

        # --- stochastic action ----------
        if self.slippery: 
            actions = [action,
                       (action + 1)%len(self.allowed_actions),
                       (action + 2)%len(self.allowed_actions),
                       (action + 3)%len(self.allowed_actions)]
            
            probs = [0.7,
                     0.1,
                     0.1,
                     0.1]
        # --------------------------------
        transitions = []
        for action, prob in zip(actions, probs):
            new_state = self.__move(state, action)
            r, c = new_state
            cell = self.grid[r][c]
            
            if cell == "G":
                reward = 1
            elif cell == "H":
                reward = 0
            else:
                reward = 0
                
            game_over = cell in ["H", "G"]
            transitions.append({"prob": prob,
                                "new_state": new_state,
                                "reward": reward,
                                "game_over": game_over})
        return transitions
        
    def n_states(self):
        return self.n_rows * self.n_cols
    
    def n_actions(self):
        return len(self.actions)    

## Testing the environment

In [115]:
# let's define grid (we can we the dynamic grid generator too)
lake_grid = [["F", "S", "F", "F"],
             ["F", "H", "F", "H"],
             ["F", "H", "F", "H"],
             ["H", "F", "F", "G"]]

In [116]:
frozen_lake = FrozenLakeEnvironment(grid=lake_grid,
                                    slippery=False)

In [117]:
frozen_lake.current_state

(0, 1)

In the above grid the position of "S" state is (0, 1), so current_state is giving the expected state coordinate.

In [118]:
# let's test the move method, if we are at (0, 0) state and try to move up, we should be at the same state
current_state = (0, 0)
action_index = frozen_lake.action_names_to_idx["up"]
frozen_lake._FrozenLakeEnvironment__move(current_state, action_index)

(0, 0)

In [119]:
# let's test the move method, if we are at (0, 0) state and try to move right, we should reach to the (0, 1) state
current_state = (0, 0)
action_index = frozen_lake.action_names_to_idx["right"]
frozen_lake._FrozenLakeEnvironment__move(current_state, action_index)

(0, 1)

`__move()` method is showing expected behaviour.

In [120]:
# Now let's test the transition prob, if we are at the Goal "G" i.e (3,3) and try to move, we are supposed to stuck to that state
frozen_lake.get_transition_prob((3, 3), action_index)

[{'prob': 1.0, 'new_state': (3, 3), 'reward': 0, 'game_over': True}]

In [121]:
# Now let's test the transition prob for near goal state
frozen_lake.get_transition_prob((3, 2), action_index)

[{'prob': 1.0, 'new_state': (3, 3), 'reward': 1, 'game_over': True}]

So far we have build a frozen lake environment if anything needs to add later, we will do that for for now we are good to go stright to the policy iteration algorithm implementation.

# Value Iteration Algorithm Implementation

## Value Iteration algorithm

- Note that the policy, action-value function, and state-value function are generally represented as 1D lists. In this implementation, we simply represent them as 2D arrays to avoid repeatedly converting between row and column indices and a 1D representation.
- A 1D list is practically convenient when actions are deterministic and the transition matrix is represented as a 2D array, where rows represent states and columns represent actions.

In [122]:
def value_update(environment, V, gamma=0.99, theta=1e-8):
    while True:
        delta = 0
        for r in range(len(V)):
            for c in range(len(V[0])):
                if environment.grid[r][c] in ["H", "G"]:
                    continue

                action_values = []
                for action in environment.allowed_actions:
                    q = 0
                    for transition in environment.get_transition_prob((r, c), action):
                        nr, nc = transition["new_state"]
                        q += transition["prob"] * (transition["reward"] + gamma * V[nr][nc])
                    
                    action_values.append(q)
                    
                new_v = np.max(action_values).item()
                delta = max(delta, abs(V[r][c] - new_v))
                V[r][c] = new_v
        if delta < theta:
            break
    return V

In [123]:
n_rows, n_cols = frozen_lake.n_rows, frozen_lake.n_cols

policy = np.zeros((n_rows, n_cols))
V = np.zeros((n_rows, n_cols))

In [124]:
value_update(frozen_lake, V)

array([[0.95099005, 0.96059601, 0.970299  , 0.96059601],
       [0.94148015, 0.        , 0.9801    , 0.        ],
       [0.93206535, 0.        , 0.99      , 0.        ],
       [0.        , 0.99      , 1.        , 0.        ]])

In [125]:
def policy_update(environment, V, gamma=0.99):
    new_policy = np.zeros((environment.n_rows, environment.n_cols), dtype=np.int8)
    for r in range(len(V)):
        for c in range(len(V[0])):
            # if environment.grid[r][c] in ["H", "G"]:
            #     # terminal state; we can ignore the policy for these states
            #     continue
            # state => (r, c)
            action_values = []
            for action in environment.allowed_actions:
                q = 0
                for transition in environment.get_transition_prob((r, c), action):
                    nr, nc = transition["new_state"]
                    q += transition["prob"] * (transition["reward"] + gamma * V[nr][nc])
                action_values.append(q)
                
            new_policy[r][c] = np.argmax(action_values).item()
    return new_policy

let's do the sanity test of policy_evaluation() implementatation to make sure there is no syntax error

In [126]:
n_rows, n_cols = frozen_lake.n_rows, frozen_lake.n_cols

policy = np.zeros((n_rows, n_cols))
V = np.zeros((n_rows, n_cols))
policy

array([[0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.]])

In [127]:
policy_update(frozen_lake, policy, V)

array([[ 0,  0,  0,  0],
       [ 0,  0,  0,  0],
       [ 0,  0,  0,  0],
       [ 0,  0, 32,  0]], dtype=int8)

In [128]:
def value_iteration(environment):
    n_rows = environment.n_rows
    n_cols = environment.n_cols

    V = np.zeros((n_rows, n_cols), dtype=np.float32)

    V = value_update(environment, V)
    policy = policy_update(environment, V)
        
    return policy, V, [policy], [V]

In [129]:
# run the policy iteration function to find the best greedy policy
policy, V, policy_history, V_history = value_iteration(frozen_lake)

In [130]:
len(policy_history)

1

In [131]:
policy

array([[2, 2, 1, 0],
       [3, 0, 1, 0],
       [3, 0, 1, 0],
       [0, 2, 2, 0]], dtype=int8)

In [132]:
text_policy = []
for r in range(policy.shape[0]):
    col = []
    for c in range(policy.shape[1]):
        col.append(ACTION_IDX_TO_NAME[policy[r][c].item()])
    text_policy.append(col)

In [133]:
text_policy

[['right', 'right', 'down', 'left'],
 ['up', 'left', 'down', 'left'],
 ['up', 'left', 'down', 'left'],
 ['left', 'right', 'right', 'left']]

# Render policy
Let's create a rendering function to visualize the policy clearly.

In [134]:
import matplotlib.pyplot as plt

In [135]:
ARROWS = {
    0: "←",
    1: "↓",
    2: "→",
    3: "↑"
}

In [136]:
import pandas as pd
from IPython.display import display

def render_policy_and_value(env, policy, V=None):
    """
    Render the policy and optionally the state-value function in a 
    visually appealing grid format using emojis and arrows.

    Args:
        env: Frozen Lake environment object with `grid`, `n_rows`, `n_cols`
        policy: 2D array of actions for each state
        V: 2D array of state values (optional)
    """
    rows, cols = env.n_rows, env.n_cols
    
    # Icons and arrows
    icons = {"S": "🚀", "H": "🕳️", "G": "🏁", "F": ""}
    arrows = {0: "←", 1: "↓", 2: "→", 3: "↑"}

    # --- Policy Display ---
    grid_policy = []
    for r in range(rows):
        row_display = []
        for c in range(cols):
            tile = env.grid[r][c]
            if tile in ["S", "F"]:
                action = policy[r][c]
                content = f"{icons[tile]} {arrows[action]}"
            else:
                content = icons[tile]
            row_display.append(content)
        grid_policy.append(row_display)

    df_policy = pd.DataFrame(grid_policy)
    
    # --- Value Function Display ---
    if V is not None:
        grid_value = []
        for r in range(rows):
            row_display = []
            for c in range(cols):
                val = V[r][c]
                row_display.append(f"{val:.2f}")
            grid_value.append(row_display)
        df_value = pd.DataFrame(grid_value)

    # --- Styling function ---
    def style_cells(val):
        style = 'width: 60px; height: 60px; text-align: center; font-size: 20px; border: 1px solid #dee2e6;'
        if "🏁" in val: return style + 'background-color: #d4edda;' # Green
        if "🕳️" in val: return style + 'background-color: #f8d7da;' # Red
        if "🚀" in val: return style + 'background-color: #cce5ff;' # Blue
        return style + 'background-color: #89cfef;'  # Frozen tiles

    # --- Render ---
    print("Policy:")
    display(df_policy.style.map(style_cells))
    
    if V is not None:
        print("State-Value Function:")
        display(df_value.style.set_properties(**{
            'width': '60px', 
            'height': '60px', 
            'text-align': 'center', 
            'font-size': '16px',
            'border': '1px solid #dee2e6'
        }))


In [137]:
render_policy_and_value(frozen_lake, policy, V)

Policy:


,0,1,2,3
0,→,🚀 →,↓,←
1,↑,🕳️,↓,🕳️
2,↑,🕳️,↓,🕳️
3,🕳️,→,→,🏁


State-Value Function:


,0,1,2,3
0,0.95,0.96,0.97,0.96
1,0.94,0.00,0.98,0.00
2,0.93,0.00,0.99,0.00
3,0.00,0.99,1.00,0.00


# Animate policy and state value history

In [138]:
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import numpy as np
from IPython.display import HTML

In [139]:
def animate_policy_value_video(env, policy_history, V_history=None, interval=500):
    """
    Animate the evolution of policy and state-value function like a video.

    Args:
        env: Frozen Lake environment with `grid`, `n_rows`, `n_cols`
        policy_history: List of 2D arrays of actions
        V_history: Optional list of 2D arrays of state values
        interval: Time between frames in milliseconds
    """
    rows, cols = env.n_rows, env.n_cols
    arrows = {0: '←', 1: '↓', 2: '→', 3: '↑'}
    
    fig, ax = plt.subplots(figsize=(cols//2, rows//2))
    
    def update(frame):
        ax.clear()
        ax.set_xticks(np.arange(cols+1)-0.5, minor=False)
        ax.set_yticks(np.arange(rows+1)-0.5, minor=False)
        ax.set_xticklabels([])
        ax.set_yticklabels([])
        ax.grid(True)
        ax.invert_yaxis()

        policy = policy_history[frame]
        if V_history is not None:
            V = V_history[frame]
            # Plot heatmap
            im = ax.imshow(V, cmap='Blues', alpha=0.6)
            for (r, c), val in np.ndenumerate(V):
                ax.text(c, r, f"{val:.1f}", ha='center', va='center', fontsize=15)
        else:
            im = None

        # Overlay policy arrows
        for r in range(rows):
            for c in range(cols):
                tile = env.grid[r][c]
                if tile in ['H', 'G', 'S']:
                    continue
                action = policy[r][c]
                ax.text(c, r, arrows[action], ha='center', va='center', fontsize=15, color='red')

        # Overlay start, goal, hole
        for r in range(rows):
            for c in range(cols):
                tile = env.grid[r][c]
                if tile == 'S':
                    ax.text(c, r, 'S', ha='center', va='center', fontsize=15)
                elif tile == 'G':
                    ax.text(c, r, 'G', ha='center', va='center', fontsize=15)
                elif tile == 'H':
                    ax.text(c, r, 'H', ha='center', va='center', fontsize=15)

        ax.set_title(f"Iteration {frame+1}")

    ani = animation.FuncAnimation(fig, update, frames=len(policy_history), interval=interval)
    plt.close(fig)  # Prevent double display in notebooks
    return ani


In [140]:
ani = animate_policy_value_video(frozen_lake, policy_history)
HTML(ani.to_jshtml())

# Let Play with different grid setup

In [141]:
grid = generate_grid_randomly(10, 10, n_holes=10)
grid

[['F', 'F', 'F', 'F', 'F', 'F', 'F', 'F', 'F', 'F'],
 ['F', 'F', 'F', 'F', 'F', 'F', 'F', 'F', 'F', 'F'],
 ['G', 'F', 'F', 'F', 'F', 'F', 'F', 'F', 'F', 'H'],
 ['F', 'F', 'F', 'F', 'F', 'F', 'F', 'F', 'F', 'F'],
 ['H', 'F', 'H', 'F', 'F', 'F', 'F', 'F', 'F', 'F'],
 ['F', 'F', 'F', 'F', 'F', 'F', 'F', 'F', 'F', 'F'],
 ['H', 'H', 'H', 'F', 'F', 'F', 'F', 'S', 'F', 'F'],
 ['F', 'H', 'F', 'F', 'F', 'F', 'F', 'F', 'F', 'F'],
 ['F', 'H', 'F', 'F', 'F', 'H', 'F', 'F', 'F', 'F'],
 ['F', 'F', 'F', 'F', 'F', 'H', 'F', 'F', 'F', 'F']]

In [142]:
frozen_lake = FrozenLakeEnvironment(grid=grid,
                                    slippery=False)
policy, V, policy_history, V_history = value_iteration(frozen_lake)

In [143]:
len(policy_history)

1

In [144]:
ani = animate_policy_value_video(frozen_lake, policy_history)
HTML(ani.to_jshtml())